In [1]:
! pip install langchain langchain-core langchain-groq python-dotenv pydantic


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Hp\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip


# String Output Parser (StrOutputParser)

In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

parser = StrOutputParser()

chain = model | parser

response = chain.invoke(
    "What is Machine Learning?"
)

print(response)
print(type(response))

**Machine Learning (ML) Definition:**
Machine Learning is a subset of Artificial Intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed.

**Key Characteristics:**

1. **Data-Driven**: ML relies on large datasets to learn patterns, relationships, and trends.
2. **Self-Improvement**: ML algorithms can improve their performance over time as they receive more data and feedback.
3. **Autonomy**: ML models can operate independently, making decisions or predictions without human intervention.

**Types of Machine Learning:**

1. **Supervised Learning**: The algorithm learns from labeled data to make predictions on new, unseen data.
2. **Unsupervised Learning**: The algorithm discovers patterns and relationships in unlabeled data.
3. **Semi-Supervised Learning**: The algorithm uses a combination of labeled and unlabeled data to learn.
4. **Reinforcement Learning**: The algorithm learns through trial and error,

# (Prompt + Parser)

In [3]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one sentence."
)

parser = StrOutputParser()

chain = prompt | model | parser

response = chain.invoke(
    {"topic": "LangChain"}
)

print(response)

LangChain is an open-source framework that enables developers to build applications on top of large language models, providing a set of tools and libraries to simplify the process of integrating AI models into software projects.


# JSON Output Parser (JsonOutputParser)

In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

parser = JsonOutputParser()

chain = model | parser

response = chain.invoke("""
Return ONLY valid JSON.

{
"name":"Rahul",
"age":25,
"city":"Pune"
}
""")

print(response)
print(type(response))

{'name': 'Rahul', 'age': 25, 'city': 'Pune'}
<class 'dict'>


# Using PromptTemplate

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = PromptTemplate(
    template="""
Return only JSON.

{format_instructions}

Question:
{query}
""",
    input_variables=["query"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

chain = prompt | model | parser

response = chain.invoke(
    {
        "query":"Generate details of one student."
    }
)

print(response)

{'student_id': 12345, 'name': 'John Doe', 'age': 20, 'grade': 'Sophomore', 'major': 'Computer Science', 'gpa': 3.5}


# Structured Output Parser

In [9]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

class Student(BaseModel):
    name: str = Field(description="Student name")
    age: int = Field(description="Student age")
    course: str = Field(description="Student course")

structured_model = model.with_structured_output(Student)

response = structured_model.invoke(
    "Rohan is 22 years old studying AI and Data Science."
)

print(response)

name='Rohan' age=22 course='AI and Data Science'


# Pydantic Output Parser

In [10]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

class Student(BaseModel):
    name: str = Field(description="Student Name")
    age: int = Field(description="Student Age")
    course: str = Field(description="Student Course")

parser = PydanticOutputParser(
    pydantic_object=Student
)

prompt = PromptTemplate(
    template="""
Answer the user query.

{format_instructions}

Query:
{query}
""",
    input_variables=["query"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

chain = prompt | model | parser

response = chain.invoke(
    {
        "query":"Amit is 21 years old studying Computer Science."
    }
)

print(response)

name='Amit' age=21 course='Computer Science'
